In [ ]:
# ============================================================
# mBART-large-50 + LoRA — Nepali Abstractive Summarization
# ============================================================
# Dependencies (install once):
#   pip install torch pandas numpy scikit-learn transformers
#               datasets sentencepiece peft accelerate
#               sacrebleu bert-score tqdm
#
# Environment:
#   NEPALI_DATA_DIR      folder containing small.csv, medium.csv, large.csv
#                        (default: ./data/)
#   NEPALI_WORK_DIR      output folder (default: ./mbart_lora_workspace/)
#   MBART_BASE_MODEL     HF model id (default: facebook/mbart-large-50)
#
# Reproducibility: split uses seed 42, deterministic cuDNN.
# Metrics: SacreBLEU (flores200, 13a), ROUGE-1/2/L (manual F1),
#          BERTScore (ne), Jaccard (word-unigram, char-bigram).
#
# Note on language token: mBART generation requires an explicit
# forced_bos_token_id; without it, mBART may generate in the wrong
# language. This file sets it to ne_NP.
# ============================================================

import os
import random
import shutil
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
)
from peft import LoraConfig, get_peft_model, TaskType
from sacrebleu.metrics import BLEU
from bert_score import score as bert_score_fn
from tqdm.auto import tqdm


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
CONFIG = {
    "data_dir":   Path(os.environ.get("NEPALI_DATA_DIR", "./data")),
    "work_dir":   Path(os.environ.get("NEPALI_WORK_DIR", "./mbart_lora_workspace")),
    "base_model": os.environ.get("MBART_BASE_MODEL", "facebook/mbart-large-50"),

    "src_lang": "ne_NP",
    "tgt_lang": "ne_NP",

    "test_size":      0.2,
    "seed":           42,
    "max_input_len":  512,
    "max_target_len": 128,

    # LoRA — unchanged
    "lora_r":       8,
    "lora_alpha":   32,
    "lora_dropout": 0.1,
    "lora_targets": ["q_proj", "v_proj"],

    # Training — unchanged
    "epochs":              3,
    "lr":                  1e-4,
    "batch_size":          4,
    "grad_accum":          8,
    "warmup_steps":        500,
    "weight_decay":        0.01,
    "fp16":                True,
    "max_grad_norm":       1.0,
    "save_total_limit":    1,
    "dataloader_workers":  4,
    "logging_steps":       500,

    # Generation — unchanged
    "gen_max_length":    128,
    "gen_num_beams":     4,
    "eval_batch_size":   8,
}


# ------------------------------------------------------------
# Reproducibility and runtime
# ------------------------------------------------------------
random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("CUDA GPU required.")

WORK_DIR        = CONFIG["work_dir"]
WORK_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR        = WORK_DIR / "mbart-lora-checkpoints"
FINAL_MODEL_DIR = WORK_DIR / "mbart-lora-final"
TEST_RAW_CSV    = WORK_DIR / "mbart_test_raw.csv"
PRED_CSV        = WORK_DIR / "mbart_test_predictions.csv"
SCORES_CSV      = WORK_DIR / "mbart_final_scores.csv"


# ------------------------------------------------------------
# Data: load, clean, split
# ------------------------------------------------------------
def safe_read(path):
    """Read CSV, falling back to latin-1 on UnicodeDecodeError."""
    try:
        return pd.read_csv(path, encoding="utf-8")
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin-1")


def load_and_split(cfg):
    data_dir = cfg["data_dir"]
    required = ("small.csv", "medium.csv", "large.csv")
    missing = [n for n in required if not (data_dir / n).is_file()]
    if missing:
        raise FileNotFoundError(
            f"Missing {missing} in {data_dir.resolve()}. "
            f"Set NEPALI_DATA_DIR or place the CSVs in ./data/."
        )

    df = pd.concat([safe_read(data_dir / n) for n in required], ignore_index=True)
    df = df.dropna(subset=["content", "summary"]).copy()
    df["content"] = df["content"].astype(str).str.strip()
    df["summary"] = df["summary"].astype(str).str.strip()
    df = df[(df["content"] != "") & (df["summary"] != "")]
    df = df.drop_duplicates(subset=["content"], keep="first").reset_index(drop=True)
    print(f"Cleaned rows: {len(df)}")

    train_df, temp_df = train_test_split(
        df, test_size=cfg["test_size"], random_state=cfg["seed"]
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.5, random_state=cfg["seed"]
    )
    print(f"Train/Val/Test: {len(train_df)}/{len(val_df)}/{len(test_df)}")
    return train_df, val_df, test_df


train_df, val_df, test_df = load_and_split(CONFIG)
test_df[["content", "summary"]].to_csv(TEST_RAW_CSV, index=False, encoding="utf-8")


# ------------------------------------------------------------
# Tokenizer and base model
# ------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(CONFIG["base_model"])
tokenizer.src_lang = CONFIG["src_lang"]
tokenizer.tgt_lang = CONFIG["tgt_lang"]
LANG_TOKEN_ID = tokenizer.lang_code_to_id[CONFIG["tgt_lang"]]
print(f"Tokenizer: {type(tokenizer).__name__}, vocab={tokenizer.vocab_size}")
print(f"src={tokenizer.src_lang} | tgt={tokenizer.tgt_lang} | forced_bos={LANG_TOKEN_ID}")

base_model = AutoModelForSeq2SeqLM.from_pretrained(CONFIG["base_model"])


# ------------------------------------------------------------
# Apply LoRA
# ------------------------------------------------------------
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    bias="none",
    target_modules=CONFIG["lora_targets"],
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
model.base_model.gradient_checkpointing_enable()


# ------------------------------------------------------------
# Tokenize
# ------------------------------------------------------------
def tokenize_fn(examples):
    """Encode source as input_ids, target as labels (mBART handles tgt_lang)."""
    inputs = tokenizer(
        examples["content"],
        max_length=CONFIG["max_input_len"],
        truncation=True,
        padding=False,
    )
    labels = tokenizer(
        text_target=examples["summary"],
        max_length=CONFIG["max_target_len"],
        truncation=True,
        padding=False,
    )
    inputs["labels"] = labels["input_ids"]
    return inputs


train_ds = Dataset.from_pandas(train_df[["content", "summary"]])
val_ds   = Dataset.from_pandas(val_df[["content", "summary"]])

tokenized_train = train_ds.map(tokenize_fn, batched=True, remove_columns=["content", "summary"])
tokenized_val   = val_ds.map(tokenize_fn, batched=True, remove_columns=["content", "summary"])


# ------------------------------------------------------------
# Training arguments and Trainer
# ------------------------------------------------------------
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

training_args = Seq2SeqTrainingArguments(
    output_dir=str(CKPT_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=CONFIG["logging_steps"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["grad_accum"],
    num_train_epochs=CONFIG["epochs"],
    learning_rate=CONFIG["lr"],
    warmup_steps=CONFIG["warmup_steps"],
    weight_decay=CONFIG["weight_decay"],
    fp16=CONFIG["fp16"],
    predict_with_generate=True,
    generation_max_length=CONFIG["gen_max_length"],
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=CONFIG["save_total_limit"],
    max_grad_norm=CONFIG["max_grad_norm"],
    dataloader_num_workers=CONFIG["dataloader_workers"],
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

print("Starting training...")
trainer.train()
print("Training finished.")


# ------------------------------------------------------------
# Merge LoRA into base and save
# ------------------------------------------------------------
merged_model = model.merge_and_unload()
merged_model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
shutil.make_archive(str(WORK_DIR / "mbart-lora-final"), "zip", FINAL_MODEL_DIR)
print(f"Saved merged model -> {FINAL_MODEL_DIR}")


# ------------------------------------------------------------
# Inference: reload merged model, batch-generate on test set
# ------------------------------------------------------------
infer_tok = AutoTokenizer.from_pretrained(FINAL_MODEL_DIR)
infer_tok.src_lang = CONFIG["src_lang"]
infer_tok.tgt_lang = CONFIG["tgt_lang"]
infer_lang_id = infer_tok.lang_code_to_id[CONFIG["tgt_lang"]]

infer_model = AutoModelForSeq2SeqLM.from_pretrained(
    FINAL_MODEL_DIR, torch_dtype=torch.float16,
).to(device).eval()


@torch.no_grad()
def generate_batch(texts):
    """Greedy/beam generation; language token set explicitly for mBART."""
    enc = infer_tok(
        texts,
        max_length=CONFIG["max_input_len"],
        truncation=True,
        padding=True,
        return_tensors="pt",
    ).to(device)
    out = infer_model.generate(
        **enc,
        forced_bos_token_id=infer_lang_id,          # mBART requires this
        max_length=CONFIG["gen_max_length"],
        num_beams=CONFIG["gen_num_beams"],
        early_stopping=True,
    )
    return infer_tok.batch_decode(out, skip_special_tokens=True)


test_df = pd.read_csv(TEST_RAW_CSV)
predictions = []
for start in tqdm(range(0, len(test_df), CONFIG["eval_batch_size"]), desc="Generating"):
    batch = test_df["content"].iloc[start:start + CONFIG["eval_batch_size"]].astype(str).tolist()
    predictions.extend(generate_batch(batch))

pred_df = pd.DataFrame({
    "source":    test_df["content"].astype(str),
    "reference": test_df["summary"].astype(str),
    "predicted": predictions,
})
pred_df.to_csv(PRED_CSV, index=False, encoding="utf-8")
print(f"Saved {len(pred_df)} predictions -> {PRED_CSV}")


# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------
pred_df = pd.read_csv(PRED_CSV).fillna("")
empty = pred_df["predicted"].astype(str).str.strip() == ""
n_empty = int(empty.sum())
if n_empty:
    pred_df.loc[empty, "predicted"] = "।"

refs = pred_df["reference"].astype(str).tolist()
hyps = pred_df["predicted"].astype(str).tolist()
print(f"Evaluating {len(refs)} predictions (empty: {n_empty})\n")

# SacreBLEU
bleu_f  = BLEU(tokenize="flores200", max_ngram_order=4).corpus_score(hyps, [refs]).score
bleu_13 = BLEU(tokenize="13a",      max_ngram_order=4).corpus_score(hyps, [refs]).score


# ROUGE (manual F1)
def _ngrams(tokens, n):
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]


def rouge_n_f1(hyp, ref, n):
    h_ng, r_ng = _ngrams(hyp, n), _ngrams(ref, n)
    if not h_ng or not r_ng:
        return 0.0
    overlap = sum((Counter(h_ng) & Counter(r_ng)).values())
    p, r = overlap / len(h_ng), overlap / len(r_ng)
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)


def rouge_l_f1(hyp, ref):
    m, n = len(hyp), len(ref)
    if m == 0 or n == 0:
        return 0.0
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            dp[i][j] = (
                dp[i - 1][j - 1] + 1 if hyp[i - 1] == ref[j - 1]
                else max(dp[i - 1][j], dp[i][j - 1])
            )
    lcs = dp[m][n]
    p, r = lcs / m, lcs / n
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)


pairs = [(h.split(), r.split()) for h, r in zip(hyps, refs)]
r1 = float(np.mean([rouge_n_f1(h, r, 1) for h, r in pairs]))
r2 = float(np.mean([rouge_n_f1(h, r, 2) for h, r in pairs]))
rL = float(np.mean([rouge_l_f1(h, r)    for h, r in pairs]))

# BERTScore
P, R, F1 = bert_score_fn(hyps, refs, lang="ne", verbose=False, device=device.type)


# Jaccard
def _jaccard(A, B):
    if not A and not B:
        return 1.0
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)


def jaccard_word(a, b):
    return _jaccard(set(str(a).split()), set(str(b).split()))


def jaccard_char_bigram(a, b):
    a, b = str(a), str(b)
    A = {a[i:i + 2] for i in range(max(len(a) - 1, 0))}
    B = {b[i:i + 2] for i in range(max(len(b) - 1, 0))}
    return _jaccard(A, B)


jac_w = float(np.mean([jaccard_word(h, r) for h, r in zip(hyps, refs)]))
jac_c = float(np.mean([jaccard_char_bigram(h, r) for h, r in zip(hyps, refs)]))


# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------
summary = {
    "model":               "mBART-large-50 + LoRA",
    "sacrebleu_flores200": round(bleu_f,  4),
    "sacrebleu_13a":       round(bleu_13, 4),
    "rouge1":              round(r1, 4),
    "rouge2":              round(r2, 4),
    "rougeL":              round(rL, 4),
    "bertscore_p":         round(P.mean().item(),  4),
    "bertscore_r":         round(R.mean().item(),  4),
    "bertscore_f1":        round(F1.mean().item(), 4),
    "jaccard_word":        round(jac_w, 4),
    "jaccard_char":        round(jac_c, 4),
    "n_samples":           len(refs),
    "n_empty":             n_empty,
}

print("=" * 55)
print("  mBART-large-50 + LoRA — FINAL METRICS")
print("=" * 55)
for k, v in summary.items():
    print(f"  {k:<22}: {v}")
print("=" * 55)
pd.Series(summary).to_csv(SCORES_CSV)
print(f"Saved -> {SCORES_CSV}")